# run_pick_place_trials: success rates vs the README targets

Run notebook (real arm or Gazebo). Milestones: pick-and-place success on the first attempt
**55-60 %** (initial target) and after automatic verify-and-retry **95 %** (proposed); per-move
success >= 99 % (stretch). A trial = pick from a square + place on another; success = grasp
verified *and* the piece ends centered on the target within tolerance (camera check).

**TODO**
- [ ] fixed, seeded list of (src, dst, kind) trials covering near/far ranks and all kinds
- [ ] backend choice (GazeboBackend first, then ServoBackend)
- [ ] placement check with the camera (grasp_verify.camera_check)
- [ ] run >= 100 trials per condition; save CSV; Wilson confidence intervals in the report
- [ ] repeat for each control path (Python / C++ / STM32) for the comparison table

In [ ]:
import numpy as np
import pandas as pd
%run ./grasp_verify.ipynb

TARGET_FIRST = (0.55, 0.60)
TARGET_RETRY = 0.95
OUT_CSV = REPO_ROOT / "data" / "arm" / "pick_place_trials.csv"

def make_trials(n=100, seed=0):
    rng = np.random.default_rng(seed)
    kinds = list("PNBRQK")
    return [(ALL_SQUARES[rng.integers(64)], ALL_SQUARES[rng.integers(64)], kinds[rng.integers(6)])
            for _ in range(n)]

def wilson(k, n, z=1.96):
    if n == 0:
        return (np.nan, np.nan)
    p = k / n
    c = (p + z * z / (2 * n)) / (1 + z * z / n)
    h = z * np.sqrt(p * (1 - p) / n + z * z / (4 * n * n)) / (1 + z * z / n)
    return (c - h, c + h)

In [ ]:
def run_trials(backend, trials, policy=RetryPolicy()):
    """TODO: for each (src, dst, kind): reset the piece onto src (human or Gazebo teleport),
    pick_with_retry, then place + placement check; record attempts, first_ok, final_ok,
    placement error (mm), time. Skip squares the planner cannot reach (record as 'unreachable')."""
    raise NotImplementedError

def report(df):
    n = len(df)
    k1, k2 = int(df.first_ok.sum()), int(df.final_ok.sum())
    return pd.DataFrame({
        "rate": [k1 / n, k2 / n],
        "ci95": [wilson(k1, n), wilson(k2, n)],
        "target": [f"{TARGET_FIRST[0]:.0%}-{TARGET_FIRST[1]:.0%}", f"{TARGET_RETRY:.0%}"],
    }, index=["first attempt", "after retry"])

# df = run_trials(GazeboBackend(), make_trials()); df.to_csv(OUT_CSV, index=False); report(df)